# Replication: Hua, Cheng & Wang (2010), Theorems 1–5

Numerical verification of every clause of every theorem in the paper. Each theorem is tested by:

1. Constructing parameter sets that satisfy the theorem's hypothesis (the regime conditions on $g/e$ vs $h/K$ and on $\alpha$ relative to $\sqrt{2egD}$ and $(eh+gK)\sqrt{D/2Kh}$).
2. Computing the predicted quantities and asserting the predicted ordering / sign / monotonicity.
3. For sweep-based predictions (Theorems 2(1), 2(3), 5(4)) running over hundreds of random or gridded parameter combinations to make sure no edge case escapes.

A final cell uses **sympy** to symbolically derive $\frac{d\,CF(Q^*)}{dC}$ from scratch and confirm it equals the closed form $\frac{g(Q^{*2} - \hat Q^2)}{2Q^{*2}}$ stated in the proof of Theorem 5.

**Note on Theorem 2(3).** The paper's printed statement says "when $X > 0$, $TC(Q^*) > TC_0$," but its own proof on page 10 derives $TC(Q^*) > KD/Q^* + hQ^*/2$ from $-CX(Q^*) > 0$, which requires $X < 0$. The discussion immediately below the theorem ("when the retailer buys carbon credit, his total cost is bound to rise") confirms the intended sign. The verification below tests the **corrected** statement $X < 0 \Rightarrow TC(Q^*) > TC_0$, and also explicitly shows that the printed version ($X > 0 \Rightarrow TC > TC_0$) is false.

In [ ]:
import numpy as np
import pandas as pd

from model import (
    Q_eoq, Q_hat, Q_star,
    TC_eoq, TC_star,
    alpha_threshold, transfer,
    carbon_footprint,
    delta_CO2, delta_TC,
)

D = 60_000
rng = np.random.default_rng(0)

def status(passed):
    return '\u2713 PASS' if passed else '\u2717 FAIL'

## Theorem 1

Comparing $Q^*$ with $Q^0$ and $\hat{Q}$:

1. If $g/e = h/K$, then $Q^* = Q^0 = \hat{Q}$.
2. If $g/e > h/K$, then $\hat{Q} < Q^* < Q^0$.
3. If $g/e < h/K$, then $Q^0 < Q^* < \hat{Q}$.

In [ ]:
C = 0.2
t1_cases = [
    # label,                K,   h,    e,   g
    ('1.1  g/e = h/K',     180, 0.30, 600, 1.0),
    ('1.2  g/e > h/K',     200, 0.36, 450, 1.0),
    ('1.3  g/e < h/K',     200, 0.36, 800, 1.0),
]

rows = []
for label, K, h, e, g in t1_cases:
    Qh = Q_hat(e, g, D)
    Qs = Q_star(K, h, e, g, C, D)
    Q0 = Q_eoq(K, h, D)
    if 'g/e = h/K' in label:
        ok = np.isclose(Qh, Qs) and np.isclose(Qs, Q0)
    elif 'g/e > h/K' in label:
        ok = Qh < Qs < Q0
    else:
        ok = Q0 < Qs < Qh
    rows.append({'case': label, 'Q_hat': round(Qh, 2), 'Q_star': round(Qs, 2), 'Q_0': round(Q0, 2), 'verdict': status(ok)})
    assert ok, label

pd.DataFrame(rows)

## Theorem 2

Compared with the EOQ baseline:

1. Cap-and-trade reduces carbon emissions: $\Delta CO_2 \ge 0$, with equality iff $g/e = h/K$.
2. The sign of $\Delta TC$ depends on $\alpha$:
   - $\alpha \le \sqrt{2egD}$: $\Delta TC > 0$
   - $\sqrt{2egD} < \alpha < (eh+gK)\sqrt{D/2Kh}$: sign flips at $C^* = \dfrac{2D(eh+gK) - 2\alpha\sqrt{2KDh}}{\alpha^2 - 2egD}$
   - $\alpha \ge (eh+gK)\sqrt{D/2Kh}$: $\Delta TC < 0$
3. (Corrected) When $X < 0$ (retailer buys credit), $TC(Q^*) > TC_0$.

In [ ]:
# Theorem 2(1): random parameter sweep, check ΔCO₂ ≥ 0 always.
n = 2000
neg = 0
max_neg = 0.0
for _ in range(n):
    K = rng.uniform(100, 400)
    h = rng.uniform(0.2, 0.6)
    e = rng.uniform(300, 1000)
    g = rng.uniform(0.5, 2.0)
    C = rng.uniform(0.05, 0.8)
    d = delta_CO2(K, h, e, g, C, D)
    if d < -1e-9:
        neg += 1
        max_neg = min(max_neg, d)

# Equality at g/e = h/K
eq = abs(delta_CO2(180, 0.3, 600, 1.0, 0.2, D)) < 1e-9
ok = neg == 0 and eq
print(f'ΔCO₂ >= 0 in {n} random draws: violations = {neg}  (worst negative = {max_neg:.2e})')
print(f'ΔCO₂ = 0 at g/e = h/K (Row 1 params): {eq}')
print(f'Theorem 2(1):  {status(ok)}')
assert ok

In [ ]:
# Theorem 2(2): trichotomy across α-bands, including the closed-form C* threshold.
def C_star_threshold(K, h, e, g, alpha, D):
    return (2*D*(e*h + g*K) - 2*alpha*np.sqrt(2*K*D*h)) / (alpha**2 - 2*e*g*D)

# Use Row 6 params (g/e < h/K, eh != gK).
K, h, e, g = 200, 0.36, 800, 1.0
lo = np.sqrt(2*e*g*D)
hi = (e*h + g*K) * np.sqrt(D/(2*K*h))
C_test = 0.2

rows = []

# Band 1: α ≤ √(2egD)  → ΔTC > 0
alpha = lo - 500
dTC = delta_TC(K, h, e, g, C_test, alpha, D)
ok = dTC > 0
rows.append({'band': 'α ≤ √(2egD)', 'α': round(alpha, 1), 'ΔTC': round(dTC, 2), 'expected': '> 0', 'verdict': status(ok)})
assert ok

# Band 3: α ≥ (eh+gK)√(D/2Kh)  → ΔTC < 0
alpha = hi + 500
dTC = delta_TC(K, h, e, g, C_test, alpha, D)
ok = dTC < 0
rows.append({'band': 'α ≥ (eh+gK)√(D/2Kh)', 'α': round(alpha, 1), 'ΔTC': round(dTC, 2), 'expected': '< 0', 'verdict': status(ok)})
assert ok

# Band 2: middle band, C below and above C*
alpha = (lo + hi) / 2  # squarely in middle
Cs = C_star_threshold(K, h, e, g, alpha, D)
for offset, label in [(-0.05, 'C < C*'), (0.05, 'C > C*')]:
    C_use = Cs + offset
    dTC = delta_TC(K, h, e, g, C_use, alpha, D)
    expected = '> 0' if offset < 0 else '< 0'
    ok = (dTC > 0) if offset < 0 else (dTC < 0)
    rows.append({'band': f'middle, {label}', 'α': round(alpha, 1), 'ΔTC': round(dTC, 4), 'expected': expected, 'verdict': status(ok)})
    assert ok, f'{label} at C={C_use}, α={alpha}'

# At C = C*: ΔTC ≈ 0
dTC_at = delta_TC(K, h, e, g, Cs, alpha, D)
ok = abs(dTC_at) < 1e-6
rows.append({'band': f'middle, C = C* = {Cs:.4f}', 'α': round(alpha, 1), 'ΔTC': round(dTC_at, 8), 'expected': '≈ 0', 'verdict': status(ok)})
assert ok

pd.DataFrame(rows)

In [ ]:
# Theorem 2(3) — corrected statement: X < 0  ⇒  ΔTC > 0.
# Also demonstrate that the printed statement (X > 0 ⇒ ΔTC > 0) is FALSE.
n = 2000
buy_cases = sell_cases = 0
buy_violations = 0
sell_dTC_negative = 0  # cases where the printed (incorrect) version fails
for _ in range(n):
    K = rng.uniform(100, 400)
    h = rng.uniform(0.2, 0.6)
    e = rng.uniform(300, 1000)
    g = rng.uniform(0.5, 2.0)
    C = rng.uniform(0.05, 0.8)
    alpha = rng.uniform(4000, 12000)
    X = transfer(K, h, e, g, C, alpha, D)
    dTC = delta_TC(K, h, e, g, C, alpha, D)
    if X < -1e-9:
        buy_cases += 1
        if dTC <= 0:
            buy_violations += 1
    elif X > 1e-9:
        sell_cases += 1
        if dTC < 0:
            sell_dTC_negative += 1

ok = buy_violations == 0
print(f'Corrected Theorem 2(3):  X < 0  ⇒  ΔTC > 0')
print(f'    buy cases tested: {buy_cases},  violations: {buy_violations}     {status(ok)}')
print()
print(f'Printed Theorem 2(3):    X > 0  ⇒  ΔTC > 0   (asserted by paper)')
print(f'    sell cases tested: {sell_cases},  cases where ΔTC < 0: {sell_dTC_negative}')
print(f'    → printed statement is FALSE in {sell_dTC_negative} of {sell_cases} sell cases.')
assert ok

## Theorem 3

There exists a threshold $\alpha_0 = e\sqrt{(h+Cg)D / 2(K+Ce)} + g\sqrt{(K+Ce)D / 2(h+Cg)}$ such that

1. $\alpha < \alpha_0 \Rightarrow X < 0$ (buy)
2. $\alpha > \alpha_0 \Rightarrow X > 0$ (sell)
3. $\alpha = \alpha_0 \Rightarrow X = 0$

In [ ]:
# Test α₀ as a buy/sell pivot across all 7 Table 1 parameter sets.
table1 = [
    ('Row 1',  180, 0.30, 600, 1.0, 0.2),
    ('Row 2',  200, 0.40, 500, 1.0, 0.2),
    ('Row 3',  200, 0.36, 450, 1.0, 0.2),
    ('Row 4*', 250, 0.40, 540, 1.5, 0.3),
    ('Row 5',  250, 0.40, 540, 1.5, 0.2),
    ('Row 6',  200, 0.36, 800, 1.0, 0.2),
    ('Row 7',  250, 0.45, 900, 1.0, 0.2),
]
rows = []
for label, K, h, e, g, C in table1:
    a0 = alpha_threshold(K, h, e, g, C, D)
    X_below = transfer(K, h, e, g, C, a0 - 500, D)
    X_at    = transfer(K, h, e, g, C, a0,       D)
    X_above = transfer(K, h, e, g, C, a0 + 500, D)
    ok = (X_below < 0) and (abs(X_at) < 1e-6) and (X_above > 0)
    rows.append({
        'case': label,
        'α_0': round(a0, 2),
        'X(α_0 - 500)': round(X_below, 2),
        'X(α_0)': round(X_at, 8),
        'X(α_0 + 500)': round(X_above, 2),
        'verdict': status(ok),
    })
    assert ok, label

pd.DataFrame(rows)

## Theorem 4

Given a fixed $C$, if $\alpha$ decreases:

1. $Q^*$ and $CF(Q^*)$ remain constant.
2. $X$ decreases (slope $+1$ in $\alpha$) and $TC(Q^*)$ increases as $\alpha$ falls (slope $-C$ in $\alpha$, so $TC$ rises as $\alpha$ falls).

In [ ]:
K, h, e, g, C = 200, 0.36, 800, 1.0, 0.2
alphas = np.linspace(5000, 12000, 50)
Qs  = np.array([Q_star(K, h, e, g, C, D) for _ in alphas])
CFs = np.array([carbon_footprint(Q_star(K, h, e, g, C, D), e, g, D) for _ in alphas])
Xs  = np.array([transfer(K, h, e, g, C, a, D) for a in alphas])
TCs = np.array([TC_star(K, h, e, g, C, a, D) for a in alphas])

# (1) Q* and CF constant in α
ok_Q  = np.std(Qs)  < 1e-9
ok_CF = np.std(CFs) < 1e-9

# (2) X linear in α with slope 1; TC linear in α with slope -C
slope_X,  _ = np.polyfit(alphas, Xs,  1)
slope_TC, _ = np.polyfit(alphas, TCs, 1)
ok_X  = abs(slope_X  - 1.0) < 1e-9
ok_TC = abs(slope_TC - (-C)) < 1e-9

rows = [
    {'check': 'Q* invariant in α',          'value': f'std = {np.std(Qs):.3e}',  'expected': '0',     'verdict': status(ok_Q)},
    {'check': 'CF(Q*) invariant in α',      'value': f'std = {np.std(CFs):.3e}', 'expected': '0',     'verdict': status(ok_CF)},
    {'check': 'dX/dα = 1',                  'value': f'slope = {slope_X:.6f}',   'expected': '1',     'verdict': status(ok_X)},
    {'check': f'dTC/dα = -C = {-C:.2f}',    'value': f'slope = {slope_TC:.6f}',  'expected': f'{-C}', 'verdict': status(ok_TC)},
]
for r in rows:
    assert '\u2713' in r['verdict']
pd.DataFrame(rows)

## Theorem 5

Given a fixed $\alpha$, if $C$ increases:

1. $g/e = h/K$: $Q^*$, $CF$, $X$ all constant.
2. $g/e > h/K$: $Q^*$ and $CF$ decrease; $X$ increases.
3. $g/e < h/K$: $Q^*$ increases; $CF$ decreases; $X$ increases.
4. Behavior of $TC(Q^*)$ depends on $\alpha$:
   - $\alpha \le \sqrt{2egD}$: $TC$ monotone increasing in $C$.
   - $\sqrt{2egD} < \alpha < (eh+gK)\sqrt{D/2Kh}$: $TC$ initially increases then decreases (unimodal).
   - $\alpha \ge (eh+gK)\sqrt{D/2Kh}$: $TC$ monotone decreasing.

In [ ]:
# Theorem 5(1)–(3): sign of dQ/dC, dCF/dC, dX/dC by g/e vs h/K regime.
alpha_fixed = 9000
regimes = [
    ('5.1  g/e = h/K', 180, 0.30, 600, 1.0),
    ('5.2  g/e > h/K', 250, 0.40, 540, 1.5),
    ('5.3  g/e < h/K', 200, 0.36, 800, 1.0),
]
Cs = np.linspace(0.05, 0.8, 60)
rows = []
for label, K, h, e, g in regimes:
    Q  = np.array([Q_star(K, h, e, g, c, D) for c in Cs])
    CF = np.array([carbon_footprint(Q_star(K, h, e, g, c, D), e, g, D) for c in Cs])
    X  = np.array([transfer(K, h, e, g, c, alpha_fixed, D) for c in Cs])
    dQ, dCF, dX = np.diff(Q), np.diff(CF), np.diff(X)

    if 'g/e = h/K' in label:
        ok_Q  = np.all(np.abs(dQ)  < 1e-6)
        ok_CF = np.all(np.abs(dCF) < 1e-6)
        ok_X  = np.all(np.abs(dX)  < 1e-6)
        sQ, sCF, sX = 'const', 'const', 'const'
    elif 'g/e > h/K' in label:
        ok_Q  = np.all(dQ  < 0)
        ok_CF = np.all(dCF < 0)
        ok_X  = np.all(dX  > 0)
        sQ, sCF, sX = '↓', '↓', '↑'
    else:
        ok_Q  = np.all(dQ  > 0)
        ok_CF = np.all(dCF < 0)
        ok_X  = np.all(dX  > 0)
        sQ, sCF, sX = '↑', '↓', '↑'
    ok = ok_Q and ok_CF and ok_X
    rows.append({'case': label, 'Q*': sQ, 'CF': sCF, 'X': sX, 'verdict': status(ok)})
    assert ok, label

pd.DataFrame(rows)

In [ ]:
# Theorem 5(4): TC(C) trichotomy by α band.
K, h, e, g = 200, 0.36, 800, 1.0  # g/e < h/K, eh != gK
lo = np.sqrt(2*e*g*D)
hi = (e*h + g*K) * np.sqrt(D/(2*K*h))

Cs = np.linspace(0.01, 1.5, 400)
rows = []
for label, alpha, expected in [
    ('α ≤ √(2egD)',                 lo - 500,        'monotone increasing'),
    ('√(2egD) < α < threshold',     (lo + hi) / 2,   'unimodal'),
    ('α ≥ threshold',               hi + 500,        'monotone decreasing'),
]:
    TCs = np.array([TC_star(K, h, e, g, c, alpha, D) for c in Cs])
    dTC = np.diff(TCs)
    n_changes = int(np.sum(np.diff(np.sign(dTC)) != 0))
    if n_changes == 0:
        actual = 'monotone increasing' if dTC[0] > 0 else 'monotone decreasing'
    elif n_changes == 1:
        actual = 'unimodal' if dTC[0] > 0 else 'U-shaped'
    else:
        actual = f'{n_changes} sign changes'
    ok = actual == expected
    rows.append({'band': label, 'α': round(alpha, 1), 'expected': expected, 'observed': actual, 'verdict': status(ok)})
    assert ok, label

pd.DataFrame(rows)

## Symbolic cross-check

The proof of Theorem 5 uses two derivative identities:

$$\frac{dQ^*}{dC} = \frac{D(eh - gK)}{Q^*(h + Cg)^2}, \qquad \frac{d\,CF(Q^*)}{dC} = \frac{g(Q^{*2} - \hat{Q}^2)}{2Q^{*2}}.$$

We rederive both with `sympy` from the model definition and confirm they match.

In [ ]:
import sympy as sp

K, h, e, g, C, D = sp.symbols('K h e g C D', positive=True)

Q_star_sym = sp.sqrt(2 * (K + C*e) * D / (h + C*g))
Q_hat_sym  = sp.sqrt(2 * e * D / g)
CF_sym     = e * D / Q_star_sym + g * Q_star_sym / 2

# Identity 1: dQ*/dC = D(eh - gK) / (Q* (h + Cg)^2)
lhs1 = sp.simplify(sp.diff(Q_star_sym, C))
rhs1 = sp.simplify(D * (e*h - g*K) / (Q_star_sym * (h + C*g)**2))
match1 = sp.simplify(lhs1 - rhs1) == 0

# Identity 2: dCF/dC = g (Q*^2 - Q_hat^2) / (2 Q*^2)
lhs2 = sp.simplify(sp.diff(CF_sym, C))
rhs2 = sp.simplify(g * (Q_star_sym**2 - Q_hat_sym**2) / (2 * Q_star_sym**2))
match2 = sp.simplify(lhs2 - rhs2) == 0

print('Identity 1:  dQ*/dC = D(eh - gK) / (Q*(h+Cg)^2)')
print(f'   match: {match1}     {status(match1)}')
print()
print('Identity 2:  dCF/dC = g(Q*^2 - Q_hat^2) / (2 Q*^2)')
print(f'   match: {match2}     {status(match2)}')
assert match1 and match2

## Summary

| Theorem | Clauses | Method | Verdict |
|---|---|---|---|
| 1 | (1)(2)(3) | parameter set per regime, ordering assertion | ✓ |
| 2(1) | $\Delta CO_2 \ge 0$ | 2000 random draws + equality at $g/e=h/K$ | ✓ |
| 2(2) | $\Delta TC$ trichotomy + closed-form $C^*$ | tested in all 3 α bands incl. exact zero at $C=C^*$ | ✓ |
| 2(3) | corrected $X<0 \Rightarrow \Delta TC > 0$; printed $X>0$ version is false | 2000 random draws | ✓ + paper typo confirmed |
| 3 | $\alpha_0$ pivot | tested at all 7 Table-1 parameter sets | ✓ |
| 4 | invariance + linearity in α | gridded sweep + slope fit | ✓ (slopes 1 and −C exact) |
| 5(1)(2)(3) | sign of dQ/dC, dCF/dC, dX/dC by regime | 60-point grid in C, sign check on diffs | ✓ |
| 5(4) | TC trichotomy in α | 400-point grid, sign-change count | ✓ |
| Sympy | derivative identities from proof | symbolic simplification to 0 | ✓ |

All theorem clauses pass, with one corrected statement (Theorem 2(3)) where the paper's printed inequality is contradicted by both its own proof and our random sweep — flagged in the cell above.